> **Note (September 2026).** This notebook is the record of the original hyperparameter search; the tuned values it found are listed in Supplementary Tables 2 and 4 of the paper. Its outputs have been cleared because the test-set results it printed are superseded: the numbers reported in the paper come from `model_comparison/pipeline.py` at the top of the repository, and the trained models are in `models/`. See the repository README (section *Changes*) for why.

##### 1. Load Libraries

In [ ]:
import pandas as pd
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    learning_curve,
    StratifiedKFold
)
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import numpy as np
import os
import json
import pickle
try:
    import optuna
except ImportError:
    optuna = None

##### 2. Load and Add Target Columns
In this cell, data is read from the csv files in the "data" folder and imported as dataframes in Python. The target column is added, then encoded to a numerical value for the model. The target map is exported to keep track of which classification value is associated with which species. This export can be turned on/off based on the argument given to the function (shown in the code comment below). 

In [ ]:
# dictionary for value insertion for target column
targets = {
    "PAO1": "PAO1",
    "STAPH": "Staph",
    "Corn": "Corn",
    "E.coli": "E.coli",
    "Malt": "Malt",
    "FAE": "FAE"
}

def read_all_csvs_to_df(folder_path, export_target_map=False):
    dataframes = []

    # walk the folder recursively so csv's in subfolders (e.g. "additions") are picked up too
    file_paths = []
    for root, _, files in os.walk(folder_path):
        for f in files:
            if f.endswith(".csv"):
                file_paths.append(os.path.join(root, f))
    file_paths = sorted(file_paths)

    for i, file_path in enumerate(file_paths):
        file_name = os.path.basename(file_path)
        df = pd.read_csv(file_path, header=None)

        if i != 0:
            df = df.iloc[1:].reset_index(drop=True)

        # adding the target column values based on the target dictionary above
        target_value = next((value for key, value in targets.items() if key in file_name), None)  

        if target_value is None:
            raise ValueError(f"No target mapping found for file: {file_name}")

        dataframes.append(df.assign(target=target_value))

    # Some added files have a different number of feature columns. Keep only columns
    # shared by all files so the model never receives NaNs from column misalignment.
    combined_df = pd.concat(dataframes, ignore_index=True, join="inner").copy()

    if combined_df.drop(columns=["target"]).isna().to_numpy().any():
        raise ValueError("Feature columns contain NaN. Check CSV column alignment before training.")

    # label encoding for RF model 
    label_encoder = LabelEncoder()
    combined_df["target_encoded"] = label_encoder.fit_transform(combined_df["target"])

    # exporting target value map
    if export_target_map:
        target_map = {
            str(original): int(encoded)
            for original, encoded in zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))
        }

        json_path = os.path.join(os.getcwd(), "target_encoding_map.json")
        with open(json_path, "w") as f:
            json.dump(target_map, f, indent=4)


    return combined_df


# accessing the main data folder
folder_path = os.path.join("..", "data", "six_single_species")
df = read_all_csvs_to_df(folder_path, export_target_map=False) # CHANGE ARGUMENT TO TRUE

# for testing
#print(df.head())

##### 3. Label Data and Drop Target Column
In this cell, feature labels are changed and the target column is dropped. When importing the data from a csv into the dataframe, the feature labels (wavelengths) are changed to a numerical scale from 0 to the number of features/columns in the data. This remapping is exported as a json file to keep track of what is being looked at. This export can also be disabled through argument changes in the function. Since the logic of the previous cell required the string target column, it gets removed here instead of the previous cell.

In [ ]:
def label_data(dataframe, export_feature_map=False):
    feature_cols = [
        col for col in dataframe.columns
        if col not in ["target", "target_encoded"]
    ]

    feature_values = dataframe.loc[0, feature_cols]
    feature_labels = range(1, len(feature_cols) + 1)

    clean_df = dataframe.iloc[1:].reset_index(drop=True).copy()
    clean_df = clean_df.rename(columns=dict(zip(feature_cols, feature_labels)))

    if export_feature_map:
        feature_map = dict(zip(feature_labels, feature_values))
        json_path = os.path.join(os.getcwd(), "feature_map.json")
        with open(json_path, "w") as f:
            json.dump(feature_map, f, indent=4)

    clean_df = clean_df.drop(columns=['target'], errors="ignore")

    return clean_df


#for testing
clean_df = label_data(df, export_feature_map=False) # CHANGE ARGUMENT TO TRUE
#print(clean_df.head())
#df.to_csv("file_bad.csv")

##### 4. Training the Model
In this cell, a function to train the model is being defined. Random state can be changed to a random number if need be, 42 was just an arbitrary value chosen for testing reproducibility. Actual model training will begin later when this function is called.

In [ ]:
def train_xgboost_model(
    df: pd.DataFrame,
    target_col: str = "target_encoded",
    test_size: float = 0.2,
    val_size: float = 0.2,
    random_state: int = 42,
    n_trials: int = 5,
    cv_folds: int = 5,
    use_optuna: bool = True,
    fixed_params: dict | None = None,
    return_study: bool = False,
    save_model: bool = True,
    model_dir: str = os.path.join("..", "saved_models"),
    model_filename: str = "xg_single_combined_model.pkl"
):
    """
    Splits the dataframe into train/test sets and returns an XGBoost classifier.
    You can either tune hyperparameters with Optuna or pass fixed parameters.

    Parameters:
        df (pd.DataFrame): Input dataframe containing features and target.
        target_col (str): Name of the encoded target column.
        test_size (float): Fraction of data to use for final testing.
        val_size (float): Unused when tuning with cross-validation (kept for compatibility).
        random_state (int): Seed for reproducibility.
        n_trials (int): Number of Optuna trials.
        cv_folds (int): Number of stratified CV folds used inside the Optuna objective.
        use_optuna (bool): Whether to run Optuna hyperparameter tuning.
        fixed_params (dict | None): Precomputed XGBoost parameters to use when not tuning.
        return_study (bool): Whether to also return the Optuna study object.
        save_model (bool): Whether to save the trained model as a .pkl file.
        model_dir (str): Folder used when save_model=True. Created if it doesn't exist.
        model_filename (str): File name used when save_model=True.

    Returns:
        model: Final XGBClassifier
        X_train, X_test, y_train, y_test: Final split datasets
        study: Optuna study object, only when return_study=True
    """
    if target_col not in df.columns:
        raise ValueError(f"Target column '{target_col}' not found in dataframe.")

    X = df.drop(columns=[target_col])
    y = df[target_col]

    # Final holdout test set
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state,
        stratify=y
    )

    study = None

    if use_optuna:
        if optuna is None:
            raise ImportError("Optuna is not installed. Set use_optuna=False or install optuna.")

        def objective(trial):
            params = {
                "n_estimators": trial.suggest_int("n_estimators", 200, 1200),
                "max_depth": trial.suggest_int("max_depth", 2, 6),
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
                "min_child_weight": trial.suggest_float("min_child_weight", 1, 15),
                "subsample": trial.suggest_float("subsample", 0.5, 0.9),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 0.9),
                "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
                "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 20.0, log=True),
                "gamma": trial.suggest_float("gamma", 0.0, 5.0),
                "random_state": random_state,
                # "n_jobs": -1,         # CPU only — replaced by device="cuda"
                # "tree_method": "hist", # CPU hist — replaced by device="cuda"
                "device": "cuda",        # GPU acceleration (XGBoost 2.0+)
                "eval_metric": "mlogloss",
            }

            model = XGBClassifier(**params)

            # Stratified k-fold cross-validation on the training data gives a
            # robust, overfitting-resistant estimate of generalization accuracy.
            # This prevents Optuna from selecting params that only happen to do
            # well on a single lucky validation split.
            skf = StratifiedKFold(
                n_splits=cv_folds,
                shuffle=True,
                random_state=random_state
            )
            cv_scores = cross_val_score(
                model,
                X_train,
                y_train,
                cv=skf,
                scoring="accuracy"
            )
            return cv_scores.mean()

        study = optuna.create_study(direction="maximize")
        study.optimize(objective, n_trials=n_trials)

        best_params = study.best_params
        print("Best trial mean CV accuracy:", study.best_value)
        print("Best params:", best_params)
    else:
        best_params = fixed_params or {}
        if best_params:
            print("Using fixed params:", best_params)

    # Final tuned model
    final_model = XGBClassifier(
        **best_params,
        random_state=random_state,
        # n_jobs=-1,          # CPU only — replaced by device="cuda"
        # tree_method="hist", # CPU hist — replaced by device="cuda"
        device="cuda",        # GPU acceleration (XGBoost 2.0+)
        eval_metric="mlogloss",
    )

    final_model.fit(X_train, y_train)

    # Saving the model
    if save_model:
        os.makedirs(model_dir, exist_ok=True)
        model_path = os.path.join(model_dir, model_filename)
        with open(model_path, "wb") as f:
            pickle.dump(final_model, f)
        print(f"Model saved to: {model_path}")

    if return_study:
        return final_model, X_train, X_test, y_train, y_test, study

    return final_model, X_train, X_test, y_train, y_test

##### 5. Model Evaluation
Model evaluation functions are being defined here. The classification report export can be disabled by passing an argument to the function class. 

In [ ]:
def get_unique_export_path(file_name):
    """Return a non-overwriting file path by appending a counter if needed."""
    base_name, ext = os.path.splitext(file_name)
    candidate = file_name
    counter = 1

    while os.path.exists(candidate):
        candidate = f"{base_name}_{counter}{ext}"
        counter += 1

    return candidate

def eval_model(model, X_test, y_test, save_path="confusion_matrix.png", report_filename="classification_report.csv", export_class_report=False):
    """
    Evaluates a trained XGBoost model on the test set.

    Parameters:
        model: Trained classifier
        X_test (pd.DataFrame): Test features
        y_test (pd.Series or np.ndarray): True test labels
        save_path (str): File path for the saved confusion matrix image
        report_filename (str): File name used when export_class_report=True
        export_class_report (bool): Whether to write the classification report to CSV

    Returns:
        results (dict): Dictionary of evaluation results
    """
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    weighted_f1 = f1_score(y_test, y_pred, average="weighted")
    cm = confusion_matrix(y_test, y_pred)
    report = classification_report(y_test, y_pred, zero_division=0)
    report_dict = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
    report_df = pd.DataFrame(report_dict).transpose()
    report_export_path = get_unique_export_path(report_filename)
    if export_class_report:
        report_df.to_csv(report_export_path, index=True)

    labels = sorted(set(y_test) | set(y_pred))

    fig, ax = plt.subplots(figsize=(6, 5))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title("XGBoost Confusion Matrix")
    fig.tight_layout()

    # Uncomment if you want to save the figure
    # fig.savefig(save_path, dpi=300, bbox_inches="tight")

    plt.show()
    plt.close(fig)

    print("Accuracy:", round(accuracy, 4))
    print("Weighted F1:", round(weighted_f1, 4))
    print(f"Confusion matrix save path: {save_path}")
    print(f"Classification report CSV saved to: {report_export_path}")
    print("\nClassification Report:")
    print(report)

    results = {
        "accuracy": accuracy,
        "weighted_f1": weighted_f1,
        "confusion_matrix": cm,
        "classification_report": report,
        "classification_report_csv_path": report_export_path,
        "confusion_matrix_path": save_path
    }

    return results

##### 6. Creating Noisy Spectra
- A noisy copy of the clean dataframe is generated and concatenated with the clean dataframe before training/testing.
- The model now trains and tests on a single combined clean + noisy dataset.

In [ ]:
def add_white_noise_to_features(
    X: pd.DataFrame,
    noise_level: float = 0.05,
    random_state: int = 42,
    relative: bool = True
) -> pd.DataFrame:
    """
    Create a noisy copy of spectral feature rows by adding Gaussian white noise.

    Parameters:
        X (pd.DataFrame): Feature dataframe only (no target column).
        noise_level (float): Noise strength.
            If relative=True, this is multiplied by each row's std.
            If relative=False, this is used as the std of the noise directly.
        random_state (int): Seed for reproducibility.
        relative (bool): If True, scale noise by each row's spectral std.
                         If False, use fixed std for all features.

    Returns:
        X_noisy (pd.DataFrame): Noisy copy of X with same shape/index/columns.
    """
    rng = np.random.default_rng(random_state)
    X_values = X.to_numpy(dtype=float)

    if relative:
        row_std = X_values.std(axis=1, keepdims=True)
        # avoid zero std rows causing zero-noise outputs
        row_std = np.where(row_std == 0, 1e-8, row_std)
        noise = rng.normal(loc=0.0, scale=noise_level * row_std, size=X_values.shape)
    else:
        noise = rng.normal(loc=0.0, scale=noise_level, size=X_values.shape)

    X_noisy = X_values + noise

    return pd.DataFrame(X_noisy, columns=X.columns, index=X.index)


def make_noisy_dataframe(
    df: pd.DataFrame,
    target_col: str = "target_encoded",
    noise_level: float = 0.05,
    random_state: int = 42,
    relative: bool = True
) -> pd.DataFrame:
    """Create a noisy copy of a labeled dataframe while preserving targets."""
    target_position = df.columns.get_loc(target_col)
    feature_positions = [i for i in range(df.shape[1]) if i != target_position]
    feature_df = df.iloc[:, feature_positions]

    X_noisy = add_white_noise_to_features(
        feature_df,
        noise_level=noise_level,
        random_state=random_state,
        relative=relative
    )

    noisy_df = df.copy()
    noisy_df.iloc[:, feature_positions] = X_noisy.to_numpy()

    return noisy_df


def combine_clean_and_noisy_data(
    df: pd.DataFrame,
    target_col: str = "target_encoded",
    noise_level: float = 0.05,
    random_state: int = 42,
    relative: bool = True
) -> pd.DataFrame:
    """Return one dataframe containing the original rows plus noisy copies."""
    noisy_df = make_noisy_dataframe(
        df,
        target_col=target_col,
        noise_level=noise_level,
        random_state=random_state,
        relative=relative
    )

    return pd.concat([df, noisy_df], ignore_index=True)

##### 7. Let's run it!
This is where all the functions written above come together. If you would like to save your model and use it later on for classification, set the save_model argument to True. 

In [ ]:
def main():
    """
    Main workflow for training and evaluating the XGBoost model.
    Assumes the dataframe 'clean_df' already exists in the notebook.
    The model is trained and tested on a combined clean + noisy dataset.
    Hyperparameters are tuned with Optuna using stratified cross-validation
    to reduce overfitting.
    """
    # Fallback params, only used if use_optuna=False below.
    best_params = {
        "n_estimators": 1170,
        "max_depth": 4,
        "learning_rate": 0.06036450947667917,
        "min_child_weight": 10.934131465380219,
        "subsample": 0.8984148947112249,
        "colsample_bytree": 0.6455710149287527,
        "reg_alpha": 0.00012956833208514377,
        "reg_lambda": 0.004832706748706296,
        "gamma": 0.06535901811391526
    }

    combined_df = combine_clean_and_noisy_data(
        clean_df,
        target_col="target_encoded",
        noise_level=0.05,   # adjust this as needed
        random_state=42,
        relative=True
    )

    print("Clean rows:", len(clean_df))
    print("Combined clean + noisy rows:", len(combined_df))

    model, X_train, X_test, y_train, y_test = train_xgboost_model(
        df=combined_df,
        target_col="target_encoded",
        test_size=0.2,
        random_state=42,
        n_trials=5,
        cv_folds=5,
        use_optuna=True,
        fixed_params=best_params,
        save_model=True         # set False to skip saving the trained model
    )


    # Various methods to check for over fitting
    # ---------------------------------------------------
    # Compare training vs test performance
    # ---------------------------------------------------
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc = accuracy_score(y_test, y_test_pred)

    print("Train Accuracy:", round(train_acc, 4))
    print("Test Accuracy:", round(test_acc, 4))
    print("Accuracy Gap:", round(train_acc - test_acc, 4))

    # ---------------------------------------------------
    # Cross-validation performance
    # ---------------------------------------------------
    X_full = combined_df.drop(columns=["target_encoded"])
    y_full = combined_df["target_encoded"]

    cv_scores = cross_val_score(model, X_full, y_full, cv=5, scoring="accuracy")

    print("\nCross-Validation Accuracy Scores:", np.round(cv_scores, 4))
    print("CV Mean Accuracy:", round(cv_scores.mean(), 4))
    print("CV Std Dev:", round(cv_scores.std(), 4))

    # ---------------------------------------------------
    # Learning curve
    # ---------------------------------------------------
    train_sizes, train_scores, val_scores = learning_curve(
        model,
        X_full,
        y_full,
        cv=5,
        scoring="accuracy",
        train_sizes=np.linspace(0.1, 1.0, 5),
        n_jobs=1,
        shuffle=True,
        random_state=42
    )

    train_mean = train_scores.mean(axis=1)
    val_mean = val_scores.mean(axis=1)

    plt.figure(figsize=(7, 5))
    plt.plot(train_sizes, train_mean, marker='o', label="Training Accuracy")
    plt.plot(train_sizes, val_mean, marker='o', label="Validation Accuracy")
    plt.title("Learning Curve")
    plt.xlabel("Training Set Size")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.grid(True)
    plt.show()

    # ---------------------------------------------------
    # Combined clean + noisy test evaluation
    # ---------------------------------------------------
    print("\n=== Evaluation on Combined Clean + Noisy Test Set ===")
    combined_results = eval_model(
        model,
        X_test,
        y_test,
        report_filename="classification_report_combined_clean_noisy.csv",
        export_class_report=False
    )

    return model, combined_results, combined_df

model, combined_results, combined_df = main()